In [7]:
%load_ext autoreload
%autoreload 2

import akshare as ak
import pandas as pd
from datetime import datetime
from strategy_utils import *

t_date, target_date = get_market_dates()
print(f"--- SYSTEM INITIALIZED ---")
print(f"T-Date (Data Base): {t_date}")
print(f"Target Date (Trade): {target_date}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
--- SYSTEM INITIALIZED ---
T-Date (Data Base): 20260611
Target Date (Trade): 20260612


In [8]:
print("PIPELINE: Loading historical data and training model...")

# 1. 准备训练集
all_dates = get_recent_trade_dates(10) 
all_hist_data = []
for i in range(len(all_dates)-1):
    d_prev, d_curr = all_dates[i], all_dates[i+1]
    zt_prev = fetch_data_with_cache("zt_pool", d_prev, ak.stock_zt_pool_em, date=d_prev)
    zt_curr = fetch_data_with_cache("zt_pool", d_curr, ak.stock_zt_pool_em, date=d_curr)
    
    if zt_prev.empty or zt_curr.empty: continue
    
    zt_curr_codes = set(zt_curr["代码"].unique())
    zt_prev["Label"] = zt_prev["代码"].apply(lambda x: 1 if x in zt_curr_codes else 0)
    zt_prev["日期"] = d_prev
    all_hist_data.append(zt_prev)

train_full = pd.concat(all_hist_data) if all_hist_data else pd.DataFrame()

# 2. 模型训练 (生成 scaler 和 clf)
today_zt = fetch_data_with_cache("zt_pool", t_date, ak.stock_zt_pool_em, date=t_date)

# ==========================================
# 【新增过滤步骤 1】：清洗今日涨停基础数据，踢出垃圾股和监管黑名单
# ==========================================
today_zt = filter_valid_a_shares(today_zt)
today_zt = filter_regulatory_inquiries(today_zt, t_date, lookback_days=10)

clf, scaler = modeling_demo_2_new(today_zt, t_date, train_full)
print("RESULT: Model training completed successfully.")

PIPELINE: Loading historical data and training model...
-> [合规扫描] 正在扫描近 10 个交易日的监管层问询/关注函...


  0%|          | 0/19 [00:00<?, ?it/s]

-> [风险排除] 剔除了 1 只处于监管问询/关注期的股票。


  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

RESULT: Model training completed successfully.


In [9]:
print("PIPELINE: Running prediction and event scan...")

# 1. 预测
today_processed, X_pred = factor_engineering_for_tomorrow_new(today_zt, t_date)
X_pred_scaled = scaler.transform(X_pred.values)
today_processed["Success_Prob"] = clf.predict_proba(X_pred_scaled)[:, 1]
today_processed["Final_Score"] = today_processed.apply(
    lambda row: min(0.99, row["Success_Prob"] + apply_client_preference_bonus(row)), axis=1
)
today_processed["Success_Prob"] = today_processed["Final_Score"]

print("\n=== 全市场 ML 预测评分前 20 (筛选前) ===")
# 假设我们关注这些核心指标
display_cols = ["代码", "名称", "Success_Prob", "Factor_Lianban_Height", "流通市值_亿", "封板金额_亿"]
if not today_processed.empty:
    top_20_all = today_processed.sort_values(by="Success_Prob", ascending=False).head(20)
    print(top_20_all[display_cols])
else:
    print("今日打板池经过合规过滤后无候选标的。")

final_ml_pool = select_2_each_market(today_processed) if not today_processed.empty else pd.DataFrame()

# 2. 事件扫描
print("PIPELINE: Scanning events...")
event_df = get_event_driven_pool(t_date, target_date)

# ==========================================
# 【新增过滤步骤 2】：事件驱动池（重组/复牌）也要踢出监管黑名单
# ==========================================
event_df = filter_valid_a_shares(event_df)  
event_df = filter_regulatory_inquiries(event_df, t_date, lookback_days=10)

print(f"DEBUG: 事件池扫描及合规过滤完毕，共剩余 {len(event_df) if event_df is not None else 0} 条有效数据。")

# 3. 导出报告
if not final_ml_pool.empty:
    ml_export = final_ml_pool[["代码", "名称", "Success_Prob", "Factor_Lianban_Height", "流通市值_亿", "封板金额_亿", "Is_One_Word", "所属行业"]].copy()
    ml_export.columns = ["Ticker", "Name", "ML_Success_Prob", "Limit_Height", "Market_Cap", "Sealing_Amt", "Is_One_Word_Board", "Sector"]
else:
    ml_export = pd.DataFrame(columns=["Ticker", "Name", "ML_Success_Prob", "Limit_Height", "Market_Cap", "Sealing_Amt", "Is_One_Word_Board", "Sector"])

filename = f"明日_{target_date}.xlsx"
with pd.ExcelWriter(filename, engine='openpyxl') as writer:
    if not ml_export.empty:
        ml_export.to_excel(writer, sheet_name='ML_Momentum_Pool', index=False)
    else:
        pd.DataFrame([{"Message": "今日无符合条件的合规打板标的"}]).to_excel(
            writer, sheet_name='ML_Momentum_Pool', index=False
        )
    
    if event_df is not None and isinstance(event_df, pd.DataFrame) and not event_df.empty:
        event_df.columns = ["Ticker", "Name", "Event_Type", "Keyword", "Description", "Date"]
        event_df.to_excel(writer, sheet_name='Event_Driven_Pool', index=False)
    else:
        pd.DataFrame([{"Message": "今日无重大并购重组公告或复牌 (No major events)"}]).to_excel(
            writer, sheet_name='Event_Driven_Pool', index=False
        )

print(f"--- PROCESS COMPLETE ---")
print(f"File Saved: {filename}")

PIPELINE: Running prediction and event scan...


  0%|          | 0/2 [00:00<?, ?it/s]


=== 全市场 ML 预测评分前 20 (筛选前) ===
        代码     名称  Success_Prob  Factor_Lianban_Height  流通市值_亿  封板金额_亿
0   000631   顺发恒能      0.636367                      1   76.82    1.05
5   600141   兴发集团      0.411611                      1  393.33    3.42
29  600108   亚盛集团      0.380827                      1   70.48    0.85
21  600067   冠城新材      0.368962                      1   43.98    0.39
8   600470   六国化工      0.355206                      1   29.57    0.41
13  688167   炬光科技      0.347136                      1  440.99    4.17
1   002203   海亮股份      0.277456                      1  471.41    1.04
39  688545   兴福电子      0.275731                      1  190.49    1.32
6   603065  XD宿迁联      0.260350                      3   51.78    1.71
4   600500   中化国际      0.254386                      3  258.72    3.03
55  603678   火炬电子      0.242675                      1  294.42    2.47
31  002378   章源钨业      0.220069                      1  401.92    1.72
66  300505    川金诺      0.190200               

  0%|          | 0/12 [00:00<?, ?it/s]

-> [合规扫描] 正在扫描近 10 个交易日的监管层问询/关注函...
-> [风险排除] 剔除了 9 只处于监管问询/关注期的股票。
DEBUG: 事件池扫描及合规过滤完毕，共剩余 25 条有效数据。
--- PROCESS COMPLETE ---
File Saved: 明日_20260612.xlsx
